# DefectVision — ResNet18 특징 추출

## 공통 분할 사용

프로젝트 루트에서 실행해 `mvtec_dataset.py`, `data_split.py`, `common.py`를 불러오세요.
Colab에서는 프로젝트 폴더를 준비하고 해당 폴더로 작업 위치를 변경해야 합니다.
기존 `splits/bottle.json`도 함께 가져와야 앱과 같은 최종 목록을 사용합니다.
공통 분할은 정상 통계/검증을 약 80:20, 시드 0으로 고정하며 추가 원본이 생겨도 자동 재분할하지 않습니다.


## 1. 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 데이터 압축 해제

In [ ]:
!mkdir -p /content/mvtec
!tar -xf /content/drive/MyDrive/bottle.tar.xz -C /content/mvtec
!ls /content/mvtec/bottle

ground_truth  license.txt  readme.txt  test  train


## 3. 데이터셋 로더 (전처리: 최민준)

In [ ]:
import torch
from mvtec_dataset import MVTecDataset

# 앱과 같은 splits/카테고리.json의 이미지 목록과 전처리를 사용합니다.


## 4. 데이터셋 확인

In [ ]:
from torch.utils.data import ConcatDataset

DATA_DIR = '/content/mvtec'
CATEGORY = 'bottle'
train_ds = MVTecDataset(DATA_DIR, CATEGORY, split='train')
val_ds = MVTecDataset(DATA_DIR, CATEGORY, split='val')
test_ds = MVTecDataset(DATA_DIR, CATEGORY, split='test')
# 고정 추출기는 학습하지 않으므로 정상 전체의 특징을 함께 저장할 수 있습니다.
# PaDiM 통계에는 train_ds만 사용하고 검증 표본은 통계에 섞지 않습니다.
normal_ds = ConcatDataset([train_ds, val_ds])
print('정상 통계:', len(train_ds), '/ 정상 검증:', len(val_ds), '/ 테스트:', len(test_ds))
img, label, mask, path = train_ds[0]
print(img.shape)


## 5. ResNet18 불러오기

In [ ]:
import torchvision
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
model.eval()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## 6. 특징 추출 함수

In [ ]:
import torch.nn.functional as F

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def extract_features(x):
    x = (x - MEAN) / STD                 # ImageNet 정규화
    with torch.no_grad():                # 학습 안 함 → 빠르게
        x = model.conv1(x); x = model.bn1(x); x = model.relu(x); x = model.maxpool(x)
        f1 = model.layer1(x)             # 64채널
        f2 = model.layer2(f1)            # 128채널
        f3 = model.layer3(f2)            # 256채널 (layer4는 안 씀)
    size = f1.shape[-2:]
    f2 = F.interpolate(f2, size=size, mode='nearest')
    f3 = F.interpolate(f3, size=size, mode='nearest')
    return torch.cat([f1, f2, f3], dim=1)

print(extract_features(torch.rand(2, 3, 256, 256)).shape)

torch.Size([2, 448, 64, 64])


## 7. 특징 차원 축소

In [ ]:
D = 100
idx = torch.randperm(448, generator=torch.Generator().manual_seed(1024))[:D]

def get_embedding(x):
    return extract_features(x)[:, idx]   # (장수, 100, 64, 64)

## 8. 특징 추출 및 저장

In [ ]:
from torch.utils.data import DataLoader

def run(ds, save_name):
    loader = DataLoader(ds, batch_size=16, shuffle=False)
    embs, labels, masks, paths = [], [], [], []
    for img, label, mask, path in loader:
        embs.append(get_embedding(img))
        labels.append(label); masks.append(mask); paths += list(path)
    out = {'emb': torch.cat(embs), 'labels': torch.cat(labels),
           'masks': torch.cat(masks), 'paths': paths, 'idx': idx}
    out['split'] = train_ds.split_definition
    torch.save(out, f'/content/drive/MyDrive/{save_name}')
    print(save_name, out['emb'].shape)

run(normal_ds, f'{CATEGORY}_train_features.pt')
run(test_ds,  f'{CATEGORY}_test_features.pt')

## 9. CPU 처리 시간 측정

In [ ]:
import time, statistics

def benchmark(dim, res, warmup=5, repeat=30):
    x = torch.rand(1, 3, res, res)
    sel = idx[:dim] if dim <= D else torch.arange(448)
    for _ in range(warmup):
        extract_features(x)[:, sel]
    times = []
    for _ in range(repeat):
        t = time.perf_counter()
        extract_features(x)[:, sel]
        times.append((time.perf_counter() - t) * 1000)
    return statistics.mean(times), statistics.median(times)

print('CPU 스레드 수:', torch.get_num_threads())
for res in [256, 224]:
    for dim in [448, 100, 50]:
        m, med = benchmark(dim, res)
        print(f'해상도 {res} | 차원 {dim:3d} | 평균 {m:6.1f} ms | 중앙값 {med:6.1f} ms')

CPU 스레드 수: 1
해상도 256 | 차원 448 | 평균   97.2 ms | 중앙값   95.9 ms
해상도 256 | 차원 100 | 평균   95.6 ms | 중앙값   93.8 ms
해상도 256 | 차원  50 | 평균  104.4 ms | 중앙값   95.4 ms
해상도 224 | 차원 448 | 평균  108.0 ms | 중앙값  106.8 ms
해상도 224 | 차원 100 | 평균   69.7 ms | 중앙값   68.6 ms
해상도 224 | 차원  50 | 평균   69.8 ms | 중앙값   68.7 ms


## 10. 차원별 공분산 메모리 비교

In [ ]:
for d in [448, 100, 50]:
    mb = 64*64*d*d*4 / 1024**2
    print(f'차원 {d:3d}: 공분산 약 {mb:,.0f} MB')

차원 448: 공분산 약 3,136 MB
차원 100: 공분산 약 156 MB
차원  50: 공분산 약 39 MB
